In [12]:
import random, numpy as np, torch
MAX_LEN = 256
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [13]:
import torch, psutil

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("RAM (GB):", round(psutil.virtual_memory().total / 1e9, 1))

GPU: NVIDIA A100-SXM4-80GB
VRAM (GB): 85.2
RAM (GB): 179.4


In [14]:
import torch
import pandas as pd
import numpy as np

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

import pandas as pd
from transformers import (
	DebertaV2Tokenizer,
	DebertaV2ForSequenceClassification,
	Trainer,
	TrainingArguments
)


In [15]:
DEV_PATH  = "/content/development_processed.csv"
EVAL_PATH = "/content/evaluation_processed.csv"

df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

for df in (df_dev, df_eval):
	df["title"]   = df["title"].fillna("").astype(str)
	df["article"] = df["article"].fillna("").astype(str)


In [16]:
def build_text(df):
	return df["title"] + "\n\n" + df["article"]

df_dev["text"]  = build_text(df_dev)
df_eval["text"] = build_text(df_eval)

In [17]:
# ============================================================
class NewsDataset(torch.utils.data.Dataset):
	def __init__(self, df, tokenizer, with_labels=True, max_length=MAX_LEN):
		self.texts  = df["text"].tolist()
		self.labels = df["label"].values if with_labels else None
		self.tokenizer = tokenizer
		self.max_length = max_length

	def __len__(self):
		return len(self.texts)

	def __getitem__(self, idx):
		enc = self.tokenizer(
			self.texts[idx],
			truncation=True,
			padding="max_length",
			max_length=self.max_length,
			return_tensors="pt"
		)

		item = {k: v.squeeze(0) for k, v in enc.items()}

		if self.labels is not None:
			item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

		return item

In [18]:
MODEL_NAME = "microsoft/deberta-v3-large"

tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)

model = DebertaV2ForSequenceClassification.from_pretrained(
	MODEL_NAME,
	num_labels=7
).cuda()

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
train_ds = NewsDataset(df_dev, tokenizer, with_labels=True)
eval_ds  = NewsDataset(df_eval, tokenizer, with_labels=False)



In [20]:
args = TrainingArguments(
	output_dir=f"./deberta_seed{SEED}",
	learning_rate=2e-5,
	per_device_train_batch_size=8,
	gradient_accumulation_steps=4,
	num_train_epochs=2,
	fp16=True,

	eval_strategy="no",
	save_strategy="no",

	seed=SEED,
	data_seed=SEED,

	logging_steps=100,
	report_to="none"
)


In [10]:
trainer = Trainer(
	model=model,
	args=args,
	train_dataset=train_ds
)

trainer.train()


Step,Training Loss
100,1.423600
200,0.972300
300,0.865100
400,0.796400
500,0.771900
600,0.760700
700,0.784500
800,0.721900
900,0.735400
1000,0.724600


TrainOutput(global_step=5000, training_loss=0.6480702735900878, metrics={'train_runtime': 3103.9831, 'train_samples_per_second': 51.545, 'train_steps_per_second': 1.611, 'total_flos': 7.455347604252365e+16, 'train_loss': 0.6480702735900878, 'epoch': 2.0})

In [11]:
pred_out = trainer.predict(eval_ds)
logits = pred_out.predictions

np.save(f"logits_roberta_processed_seed{SEED}_MAXLEN{MAX_LEN}.npy", logits)

preds = logits.argmax(axis=1)

submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": preds.astype(int)
})

submission.to_csv(f"submission_roberta_processed_seed{SEED}_MAXLEN{MAX_LEN}.csv", index=False)

print(f"Saved submission_roberta_processed_seed{SEED}_MAXLEN{MAX_LEN}.csv")
print(f"Saved logits_roberta_processed_seed{SEED}_MAXLEN{MAX_LEN}.npy")

Saved submission_roberta_processed_seed1337_MAXLEN256.csv
Saved logits_roberta_processed_seed1337_MAXLEN256.npy
